In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings
import time
import json
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')

In [ ]:
# PATHS, CONSTANTS, AND SIM PARAMS  

BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR     = BASE / 'supplementary' / 'pool_simulations_databalance_full'
MATRICES_DIR = BASE / 'supplementary' / 'pool_matrices_databalance_full'
RESULTS_DIR  = BASE / 'supplementary' / 'simulation_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SIMS = 100
LOCI = ['A', 'B', 'C', 'DR', 'DQ']
EPLET_CLASSES = ['ClassI', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5]            
EQUITY_ETHCATS = [1, 2, 4, 5]    
ETH_LABELS = {1: 'Caucasian', 2: 'Afroamerican', 4: 'Latin', 5: 'Asian'}


ARRIVAL_RATE = 800 / (10 * 12)  

SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,
    'ARRIVAL_RATE':     ARRIVAL_RATE,
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    0,   
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                800,
    'k_opt': {'antigen': 0, 'allele': 0, 'eplet': 0},
    'Z':     {'antigen': 10, 'allele': 10, 'eplet': 140},
}
print('Simulation parameters (data_balance, 10 loci, ABO+DSA + Rawlsian):')
print(f"  P (pop normalizer): {SIM_PARAMS['P']}")
print(f'  ARRIVAL_RATE (global, /month): {SIM_PARAMS["ARRIVAL_RATE"]:.4f}')
print(f'  Total expected arrivals:    {SIM_PARAMS["ARRIVAL_RATE"] * SIM_PARAMS["TOTAL_TIME"]:.0f}')
print(f"  k_opt: {SIM_PARAMS['k_opt']}")
print(f"  Z:     {SIM_PARAMS['Z']}")

WARMUP_SUFFIX = '_warmup' if SIM_PARAMS.get('WARMUP_MONTHS', 0) > 0 else '_nowarmup'
print(f"\n>>> VERSION: WARMUP_MONTHS={SIM_PARAMS['WARMUP_MONTHS']} -> outputs with suffix '{WARMUP_SUFFIX}'")

SMALL_ETHCATS = {6, 7}


In [ ]:
# PER-SIM DATA LOADER 

def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)
    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values for L in LOCI}
    eplet_mm   = {cls: pd.read_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet').values for cls in EPLET_CLASSES}
    return {'pool_df': pool_df, 'compat': compat,
            'antigen_mm': antigen_mm, 'allele_mm': allele_mm, 'eplet_mm': eplet_mm}

In [ ]:
# BUILD WEIGHT MATRICES 

MAX_ANTIGEN_10LOCI = 10
MAX_ALLELE_10LOCI  = 10
MAX_EPLET_10LOCI   = 140

def build_weights_10loci(sim_data):
    am = sim_data['antigen_mm']; al = sim_data['allele_mm']; ep = sim_data['eplet_mm']
    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)
    sum_eplet   = (ep['ClassI'] + ep['DR'] + ep['DQ']).astype(np.int32)
    return {
        'antigen': (MAX_ANTIGEN_10LOCI - sum_antigen).astype(np.int32),
        'allele':  (MAX_ALLELE_10LOCI  - sum_allele).astype(np.int32),
        'eplet':   (MAX_EPLET_10LOCI   - sum_eplet).astype(np.int32),
        'score_classI': (6 - (am['A'] + am['B'] + am['C'])).astype(np.int32),
        'score_DR':     (2 - am['DR']).astype(np.int32),
        'score_DQ':     (2 - am['DQ']).astype(np.int32),
    }

In [ ]:
# GRAPH (ABO+DSA only) AND WEIGHT-ATTACH

def create_graph(waiting_indices, compat):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i != j and compat[i, j] == 1:
                G.add_edge(j, i)
    return G

def changing_resolution_weights(G, weight_matrix):
    for u, v in G.edges():
        G[u][v]['weight'] = int(weight_matrix[v, u])

In [ ]:
def optimization_with_multipliers(G, pair_ethcat, multipliers,
                                  l=3, k_quality=0, Z=10, P=800):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    valid_cycles = [c for c in total_cycles
                    if all(G[u][v]['weight'] >= k_quality
                           for u, v in zip(c, c[1:] + c[:1]))]

    G_opt = nx.DiGraph()
    if not valid_cycles:
        return G_opt, []

    m = Model('kep_optimization_weighted')
    m.setParam('OutputFlag', 0)
    x = {tuple(c): m.addVar(vtype=GRB.BINARY) for c in valid_cycles}

    def cycle_payoff(c):
        arcs = list(zip(c, c[1:] + c[:1]))
        sum_v  = sum(multipliers.get(int(pair_ethcat[v]), 1.0) for _, v in arcs)
        sum_vw = sum(multipliers.get(int(pair_ethcat[v]), 1.0) * G[u][v]['weight'] / Z
                     for u, v in arcs)
        return sum_v + (1.0 / P) * sum_vw

    m.setObjective(quicksum(x[tuple(c)] * cycle_payoff(c) for c in valid_cycles), GRB.MAXIMIZE)
    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in valid_cycles if node in c) <= 1)
    m.optimize()

    selected = []
    if m.status == GRB.OPTIMAL:
        for c in valid_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v, weight=G[u][v]['weight'])
    return G_opt, selected

In [ ]:
# RUN ONE SIMULATION WITH MULTIPLIERS — STRATIFIED ARRIVALS BY ETHNICITY


def run_simulation_weighted(sim_id, opt_resolution, compat, weights, pair_ethcat,
                            multipliers, params,
                            quality_tracking=False):
    n = compat.shape[0]
    weight_for_obj = weights[opt_resolution]
    k_opt = params['k_opt'][opt_resolution]
    Z     = params['Z'][opt_resolution]
    P_val = params['P']   

  
    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

   
    available = set(range(n))
    waiting = []
    arrival_t, departure_t = {}, {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    pool_sizes_by_eth = {e: [] for e in ETHCATS}

    arrivals_by_eth   = {e: 0 for e in ETHCATS}
    departures_by_eth = {e: 0 for e in ETHCATS}

    
    G_contrib_by_eth = {e: 0.0 for e in ETHCATS}
    G_contrib_total  = 0.0

    if quality_tracking:
        quality = {(res, e): [] for res in ('antigen','allele','eplet') for e in ETHCATS}
        quality.update({(cls, e): [] for cls in ('classI','DR','DQ') for e in ETHCATS})

    # Each pair gets a uniform arrival month 
    _arrival_month = rng_arr.integers(0, params['TOTAL_TIME'], size=n)
    _arrivals_at = {m: [] for m in range(params['TOTAL_TIME'])}
    for _p in range(n):
        _arrivals_at[int(_arrival_month[_p])].append(_p)

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        for p_int in _arrivals_at[month]:
            arrival_t[p_int] = month
            available.discard(p_int)
            waiting.append(p_int)
            deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
            e = int(pair_ethcat[p_int])
            if counting and e in arrivals_by_eth: arrivals_by_eth[e] += 1

    
        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for e_ps in (ETHCATS if counting else []):
                pool_sizes_by_eth[e_ps].append(sum(1 for w in waiting if int(pair_ethcat[w]) == e_ps))
            for _w in waiting:
                runs_participated[_w] = runs_participated.get(_w, 0) + 1
            G = create_graph(waiting, compat)
            changing_resolution_weights(G, weight_for_obj)
            G_opt, selected = optimization_with_multipliers(
                G, pair_ethcat, multipliers,
                l=params['MAX_CYCLE_LENGTH'], k_quality=k_opt, Z=Z, P=params['P'])

          
            for u, v in (G_opt.edges() if counting else []):
                contrib = 1.0 + weight_for_obj[v, u] / (P_val * Z)
                G_contrib_total += contrib
                e_v = int(pair_ethcat[v])
                if e_v in G_contrib_by_eth:
                    G_contrib_by_eth[e_v] += contrib

            if quality_tracking:
                for u, v in (G_opt.edges() if counting else []):
                    e = int(pair_ethcat[v])
                    if e not in arrivals_by_eth: continue
                    quality[('antigen', e)].append(int(weights['antigen'][v, u]))
                    quality[('allele',  e)].append(int(weights['allele'][v, u]))
                    quality[('eplet',   e)].append(int(weights['eplet'][v, u]))
                    quality[('classI',  e)].append(int(weights['score_classI'][v, u]))
                    quality[('DR',      e)].append(int(weights['score_DR'][v, u]))
                    quality[('DQ',      e)].append(int(weights['score_DQ'][v, u]))

            historial_cycles.extend(selected if counting else [])
            cycled = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled]
            for p_int in cycled: departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                e = int(pair_ethcat[p_int])
                if e in departures_by_eth:
                    departures_by_eth[e] += 1

    n_arr_tot = sum(arrivals_by_eth.values())
    n_tx_tot = sum(len(c) for c in historial_cycles)
    F_total = n_tx_tot / max(n_arr_tot, 1)
    L_total = len(historial_departures) / max(n_arr_tot, 1)
    F_per_eth = {e: sum(1 for c in historial_cycles for p in c if int(pair_ethcat[p]) == e)
                       / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    L_per_eth = {e: departures_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    G_per_eth = {e: G_contrib_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    G_total   = G_contrib_total / max(n_arr_tot, 1)

    out = {
        'sim_id': sim_id, 'opt_resolution': opt_resolution,
        'total_arrivals': n_arr_tot, 'total_transplants': n_tx_tot,
        'total_departures': len(historial_departures),
        'arrivals_by_eth': arrivals_by_eth, 'departures_by_eth': departures_by_eth,
        'F_per_eth': F_per_eth, 'L_per_eth': L_per_eth,
        'F_total': F_total, 'L_total': L_total,
        'G_per_eth': G_per_eth, 'G_total': G_total,
        'historial_cycles': historial_cycles,
        'avg_pool_size': float(np.mean(pool_sizes)) if pool_sizes else 0.0,
        'avg_pool_size_by_eth': {e: float(np.mean(pool_sizes_by_eth[e])) if pool_sizes_by_eth[e] else 0.0 for e in ETHCATS},
    }
    if quality_tracking:
        wt_by_eth = {e: [] for e in ETHCATS}
        for p in {p for c in historial_cycles for p in c}:
            if p in runs_participated:
                e = int(pair_ethcat[p])
                if e in wt_by_eth: wt_by_eth[e].append(runs_participated[p])
        out['quality'] = quality
        out['waiting_times_by_eth'] = wt_by_eth
    return out

In [ ]:
# PRELOAD

t0 = time.time()
all_compat = []
all_weights = []
all_pair_ethcat = []
for sim_id in range(N_SIMS):
    sd = load_sim_data(sim_id)
    ws = build_weights_10loci(sd)
    pe = sd['pool_df']['ETHCAT'].values.astype(int)
    all_compat.append(sd['compat'])
    all_weights.append(ws)
    all_pair_ethcat.append(pe)


In [ ]:
# OBJECTIVE FUNCTION: run 100 sims, return mean F per eth + F total + mean arrivals per eth

def objective_function(multipliers, opt_resolution):
    F_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    arr_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    F_totals = []
    G_per_eth_all = {e: [] for e in EQUITY_ETHCATS}
    G_totals = []
    for sim_id in range(N_SIMS):
        res = run_simulation_weighted(
            sim_id, opt_resolution,
            all_compat[sim_id], all_weights[sim_id], all_pair_ethcat[sim_id],
            multipliers, SIM_PARAMS, quality_tracking=False)
        for e in EQUITY_ETHCATS:
            F_per_eth_all[e].append(res['F_per_eth'][e])
            arr_per_eth_all[e].append(res['arrivals_by_eth'].get(e, 0))
        F_totals.append(res['F_total'])
        for e in EQUITY_ETHCATS:
            G_per_eth_all[e].append(res['G_per_eth'][e])
        G_totals.append(res['G_total'])
    return {
        'F_per_eth':       {e: float(np.mean(F_per_eth_all[e])) for e in EQUITY_ETHCATS},
        'F_total':         float(np.mean(F_totals)),
        'G_per_eth':       {e: float(np.mean(G_per_eth_all[e])) for e in EQUITY_ETHCATS},
        'G_total':         float(np.mean(G_totals)),
        'arrivals_by_eth': {e: float(np.mean(arr_per_eth_all[e])) for e in EQUITY_ETHCATS},
    }

In [ ]:
# RAWLSIAN SINGLE-PUSH SEARCH (manuscript Eq. 13 + Eq. 14 best criterion)
# Per iteration:
#   1. Push: identify currently-worst eth among EQUITY_ETHCATS, boost its multiplier by `step`
#   2. Re-evaluate (100 sims)
#   3. Track best by max-min F(s) (Rawlsian): max over iters of min_s F(s)
#   4. STOP when F(baseline-worst-eth) at current iter > F(entire pop) at current iter,
#      OR when no improvement in `no_improvement_limit` iters

def equity_gap(F_per_eth, F_total, arrivals_by_eth):
   
    total_arr = sum(arrivals_by_eth.values()) or 1.0
    return sum(
        (arrivals_by_eth[e] / total_arr) * abs(F_per_eth[e] - F_total)
        for e in EQUITY_ETHCATS
    )

def rawlsian_search(opt_resolution, step=0.001, max_iters=500,
                    no_improvement_limit=100, verbose=True,
                    save_history_to=None):
    multipliers = {e: 1.0 for e in EQUITY_ETHCATS}

    print(f"\n========== Rawlsian search (data_balance) — opt={opt_resolution} ==========")
    t0 = time.time()

    res = objective_function(multipliers, opt_resolution)
    F_e = res['F_per_eth']
    F_pob = res['F_total']
    G_e = res['G_per_eth']
    G_pob = res['G_total']
    arrivals = res['arrivals_by_eth']
    gap_0 = equity_gap(F_e, F_pob, arrivals)
    min_F_0 = min(F_e[e] for e in EQUITY_ETHCATS)
    min_G_0 = min(G_e[e] for e in EQUITY_ETHCATS)
    argmin_0 = min(EQUITY_ETHCATS, key=lambda e: G_e[e])    # rank by G
    baseline_worst_eth = argmin_0

    print(f"[init] F = {{{', '.join(ETH_LABELS[e]+':'+f'{F_e[e]:.4f}' for e in EQUITY_ETHCATS)}}}")
    print(f"       F entire pop = {F_pob:.4f}")
    print(f"       Equity gap (Eq.14) = {gap_0:.5f}")
    print(f"       worst = {ETH_LABELS[argmin_0]} ({min_F_0:.4f})")
    print(f"       (eval time {time.time()-t0:.1f}s)")

    history = [{
        'iter': 0, 'multipliers': dict(multipliers), 'F_per_eth': dict(F_e),
        'F_total': F_pob, 'min_F': min_F_0, 'equity_gap': gap_0,
        'eval_time': time.time() - t0,
    }]

    best = {
        'iter': 0, 'multipliers': dict(multipliers),
        'F_per_eth': dict(F_e), 'F_total': F_pob,
        'G_per_eth': dict(G_e), 'G_total': G_pob,
        'min_F': min_F_0, 'min_G': min_G_0,
        'equity_gap': gap_0,
    }
    iters_since_improvement = 0

    if G_e[baseline_worst_eth] > G_pob:
        print(f"[init] STOP at iter 0: baseline-worst ({ETH_LABELS[baseline_worst_eth]}) already > G_pob.")
        out = {'opt_resolution': opt_resolution, 'best': best, 'history': history,
               'total_time_s': time.time() - t0}
        if save_history_to:
            with open(save_history_to, 'w') as f:
                json.dump(out, f, indent=2)
        return out

    for it in range(1, max_iters + 1):
        worst_e = min(EQUITY_ETHCATS, key=lambda e: G_e[e])   # rank by G
        multipliers[worst_e] = round(multipliers[worst_e] + step, 6)

        t_iter = time.time()
        res = objective_function(multipliers, opt_resolution)
        F_e = res['F_per_eth']
        F_pob = res['F_total']
        G_e = res['G_per_eth']
        G_pob = res['G_total']
        arrivals = res['arrivals_by_eth']
        min_F = min(F_e[e] for e in EQUITY_ETHCATS)
        min_G = min(G_e[e] for e in EQUITY_ETHCATS)
        argmin_e = min(EQUITY_ETHCATS, key=lambda e: G_e[e])    # rank by G
        gap = equity_gap(F_e, F_pob, arrivals)
        iter_time = time.time() - t_iter

        # BEST by Rawlsian max-min: keep the iter that maximises min_s G_s
        if min_G > best['min_G']:
            best = {
                'iter': it, 'multipliers': dict(multipliers),
                'F_per_eth': dict(F_e), 'F_total': F_pob,
                'G_per_eth': dict(G_e), 'G_total': G_pob,
                'min_F': min_F, 'min_G': min_G,
                'equity_gap': gap,
            }
            iters_since_improvement = 0
            marker = '  <-- new best'
        else:
            iters_since_improvement += 1
            marker = ''

        history.append({
            'iter': it, 'multipliers': dict(multipliers), 'F_per_eth': dict(F_e),
            'F_total': F_pob, 'min_F': min_F, 'equity_gap': gap,
            'boosted': int(worst_e), 'eval_time': iter_time,
        })

        if verbose:
            mstr = ' '.join(f"{ETH_LABELS[e][:3]}={multipliers[e]:.3f}" for e in EQUITY_ETHCATS)
            fstr = ' '.join(f"{ETH_LABELS[e][:3]}={F_e[e]:.4f}" for e in EQUITY_ETHCATS)
            print(f"[iter {it:3d}] boosted {ETH_LABELS[worst_e]:13s}  "
                  f"F=[{fstr}]")
            print(f"           F_pob={F_pob:.4f}  min_F={min_F:.4f} ({ETH_LABELS[argmin_e][:3]})  "
                  f"gap={gap:.5f}  mult=[{mstr}]  ({iter_time:.0f}s){marker}")

        # STOP (Rawlsian): baseline-worst-eth's current G > current G_pob
        if G_e[baseline_worst_eth] > G_pob:
            print(f"[iter {it}] STOP: baseline-worst ({ETH_LABELS[baseline_worst_eth]}) G = {G_e[baseline_worst_eth]:.4f} > G_pob = {G_pob:.4f}")
            break
        if iters_since_improvement >= no_improvement_limit:
            print(f"[iter {it}] STOP: no improvement in {no_improvement_limit} iters (stagnation)")
            break

        if save_history_to and it % 5 == 0:
            with open(save_history_to, 'w') as f:
                json.dump({'opt_resolution': opt_resolution, 'best': best, 'history': history}, f, indent=2)

    total_time = time.time() - t0
    print(f"\n>>> Done. Total time: {total_time/60:.1f} min, iterations: {len(history)-1}")
    print(f">>> Best iter = {best['iter']}, equity gap = {best['equity_gap']:.5f}")
    print(f">>> Best F per eth: " + ", ".join(
        f"{ETH_LABELS[e]}={best['F_per_eth'][e]:.4f}" for e in EQUITY_ETHCATS))
    print(f">>> Best multipliers: " + ", ".join(
        f"{ETH_LABELS[e]}={best['multipliers'][e]:.3f}" for e in EQUITY_ETHCATS))

    out = {'opt_resolution': opt_resolution, 'best': best, 'history': history,
           'total_time_s': total_time}
    if save_history_to:
        with open(save_history_to, 'w') as f:
            json.dump(out, f, indent=2)
    return out

In [ ]:
# RUN — opt = antigen

results_antigen = rawlsian_search(
    'antigen', step=0.00001, max_iters=500,
    save_history_to=RESULTS_DIR / f'rawlsian_search_databalance_full_10loci_antigen{WARMUP_SUFFIX}.json'
)

In [ ]:
# RUN — opt = allele

results_allele = rawlsian_search(
    'allele', step=0.00001, max_iters=500,
    save_history_to=RESULTS_DIR / f'rawlsian_search_databalance_full_10loci_allele{WARMUP_SUFFIX}.json'
)

In [ ]:
# RUN — opt = eplet

results_eplet = rawlsian_search(
    'eplet', step=0.00001, max_iters=500,
    save_history_to=RESULTS_DIR / f'rawlsian_search_databalance_full_10loci_eplet{WARMUP_SUFFIX}.json'
)

In [ ]:
# CONSOLIDATE BEST MULTIPLIERS PER SCENARIO

best_multipliers_per_scenario = {
    'antigen': results_antigen['best']['multipliers'],
    'allele':  results_allele['best']['multipliers'],
    'eplet':   results_eplet['best']['multipliers'],
}

print('Best multipliers per scenario:')
for opt_res, m in best_multipliers_per_scenario.items():
    line = '  ' + opt_res + ': ' + ', '.join(
        f'{ETH_LABELS[e]}={m[e]:.4f}' for e in EQUITY_ETHCATS)
    print(line)

with open(RESULTS_DIR / f'rawlsian_best_multipliers_databalance_full_10loci{WARMUP_SUFFIX}.json', 'w') as f:
    json.dump(best_multipliers_per_scenario, f, indent=2)
print(f"\nSaved consolidated best multipliers.")

In [ ]:
# APPLY BEST MULTIPLIERS — run full 100-sim simulation per scenario with quality tracking

RESOLUTIONS = ('antigen', 'allele', 'eplet')

print('Running full simulation with best multipliers per scenario...')
all_results = {res: [] for res in RESOLUTIONS}
t0 = time.time()
for opt_res in RESOLUTIONS:
    m = best_multipliers_per_scenario[opt_res]
    print(f"  opt={opt_res} with multipliers {m}")
    for sim_id in range(N_SIMS):
        res = run_simulation_weighted(
            sim_id, opt_res,
            all_compat[sim_id], all_weights[sim_id], all_pair_ethcat[sim_id],
            m, SIM_PARAMS, quality_tracking=True)
        all_results[opt_res].append(res)
print(f"Done. Total time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# AGGREGATION HELPER + RESULT TABLES  

def mean_ci(values, ddof=1, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1: return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m_ = arr.mean(); s = arr.std(ddof=ddof)
    low, high = st.t.interval(conf, len(arr) - 1, loc=m_, scale=s / np.sqrt(len(arr)))
    return m_, f"{m_:.3f} [{low:.3f}; {high:.3f}]"

def build_results_table(rs):
    rows = []
    for e in ETHCATS:
        arr_e = np.mean([r['arrivals_by_eth'][e] for r in rs])
        tx_e = np.mean([r['F_per_eth'][e] * r['arrivals_by_eth'][e] for r in rs])
        F_vals = [r['F_per_eth'][e] for r in rs]
        L_vals = [r['L_per_eth'][e] for r in rs]
        _, txt_F = mean_ci(F_vals); _, txt_L = mean_ci(L_vals)
        ant_vals = [np.mean(r['quality'][('antigen', e)]) for r in rs if r['quality'][('antigen', e)]]
        all_vals = [np.mean(r['quality'][('allele',  e)]) for r in rs if r['quality'][('allele',  e)]]
        epl_vals = [np.mean(r['quality'][('eplet',   e)]) for r in rs if r['quality'][('eplet',   e)]]
        _, txt_ant = mean_ci(ant_vals); _, txt_all = mean_ci(all_vals); _, txt_epl = mean_ci(epl_vals)
        cI = [np.mean(r['quality'][('classI', e)]) for r in rs if r['quality'][('classI', e)]]
        dr_v = [np.mean(r['quality'][('DR', e)]) for r in rs if r['quality'][('DR', e)]]
        dq_v = [np.mean(r['quality'][('DQ', e)]) for r in rs if r['quality'][('DQ', e)]]
        _, txt_cI = mean_ci(cI); _, txt_dr = mean_ci(dr_v); _, txt_dq = mean_ci(dq_v)
        wt_flat = [w for r in rs for w in r['waiting_times_by_eth'][e]]
        wt_mean = np.mean(wt_flat) if wt_flat else float('nan')
        still = round(1 - np.mean(F_vals) - np.mean(L_vals), 3)
        rows.append({
            'Ethnicity(s)': e, 'Arrivals': round(arr_e, 2), 'Transplants': round(tx_e, 2),
            'F(s) (Matched)': txt_F, 'HLA(s) Antigen': txt_ant, 'HLA(s) Allele': txt_all,
            'HLA(s) Eplets': txt_epl, 'Waiting Time': mean_ci([np.mean(r['waiting_times_by_eth'][e]) for r in rs if r['waiting_times_by_eth'][e]])[1], 'Pool Size': mean_ci([r['avg_pool_size_by_eth'][e] for r in rs])[1],
            'L(s) (Left Unmatched)': txt_L, '1-F(s)-L(s) (Still in KEP)': still,
            'HLA ClassI': txt_cI, 'HLA DR': txt_dr, 'HLA DQ': txt_dq,
        })
    F_tot = [r['F_total'] for r in rs]; L_tot = [r['L_total'] for r in rs]
    _, txt_F_tot = mean_ci(F_tot); _, txt_L_tot = mean_ci(L_tot)
    arr_tot = np.mean([r['total_arrivals'] for r in rs])
    tx_tot  = np.mean([r['total_transplants'] for r in rs])
    ant_ps=[]; all_ps=[]; epl_ps=[]; cI_ps=[]; dr_ps=[]; dq_ps=[]; wt_ps=[]
    for r in rs:
        ap = [v for e in ETHCATS for v in r['quality'][('antigen', e)]]
        lp = [v for e in ETHCATS for v in r['quality'][('allele',  e)]]
        ep = [v for e in ETHCATS for v in r['quality'][('eplet',   e)]]
        cp = [v for e in ETHCATS for v in r['quality'][('classI', e)]]
        drp = [v for e in ETHCATS for v in r['quality'][('DR', e)]]
        dqp = [v for e in ETHCATS for v in r['quality'][('DQ', e)]]
        wp = [w for e in ETHCATS for w in r['waiting_times_by_eth'][e]]
        if ap: ant_ps.append(np.mean(ap))
        if lp: all_ps.append(np.mean(lp))
        if ep: epl_ps.append(np.mean(ep))
        if cp: cI_ps.append(np.mean(cp))
        if drp: dr_ps.append(np.mean(drp))
        if dqp: dq_ps.append(np.mean(dqp))
        if wp: wt_ps.append(np.mean(wp))
    _, txt_ant_t = mean_ci(ant_ps); _, txt_all_t = mean_ci(all_ps); _, txt_epl_t = mean_ci(epl_ps)
    _, txt_cI_t = mean_ci(cI_ps); _, txt_dr_t = mean_ci(dr_ps); _, txt_dq_t = mean_ci(dq_ps)
    still_t = round(1 - np.mean(F_tot) - np.mean(L_tot), 3)
    rows.append({
        'Ethnicity(s)': 'Entire Population', 'Arrivals': round(arr_tot, 2),
        'Transplants': round(tx_tot, 2), 'F(s) (Matched)': txt_F_tot,
        'HLA(s) Antigen': txt_ant_t, 'HLA(s) Allele': txt_all_t, 'HLA(s) Eplets': txt_epl_t,
        'Waiting Time': mean_ci(wt_ps)[1], 'Pool Size': mean_ci([r['avg_pool_size'] for r in rs])[1],
        'L(s) (Left Unmatched)': txt_L_tot, '1-F(s)-L(s) (Still in KEP)': still_t,
        'HLA ClassI': txt_cI_t, 'HLA DR': txt_dr_t, 'HLA DQ': txt_dq_t,
    })
    return pd.DataFrame(rows)

tables = {}
for opt_res in RESOLUTIONS:
    print(f"\n========== RESULTS (data_balance Rawlsian) — opt={opt_res} ==========\n")
    tbl = build_results_table(all_results[opt_res])
    tables[opt_res] = tbl
    display(tbl)

# Save raw Excel
out_path = RESULTS_DIR / f'results_databalance_full_10loci_3scenarios_rawlsian{WARMUP_SUFFIX}.xlsx'
with pd.ExcelWriter(out_path) as writer:
    for opt_res, tbl in tables.items():
        tbl.to_excel(writer, sheet_name=f'opt_{opt_res}', index=False)
print(f'Saved: {out_path}')

In [ ]:
# RANK TESTS PER ETHNICITY — ALL METRICS, ALL 4 ETHNICITIES  

from scipy.stats import wilcoxon, binomtest

TESTED_ETHCATS = [1, 2, 4, 5]   

def _mean_or_nan(lst):
    return float(np.mean(lst)) if len(lst) else float('nan')
def _F_eth(r, e):
    return r['F_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')
def _L_eth(r, e):
    return r['L_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

METRIC_EXTRACTORS = {
    'F(s) (Matched)':        (_F_eth, lambda r: r['F_total']),
    'L(s) (Left Unmatched)': (_L_eth, lambda r: r['L_total']),
    'HLA(s) Antigen': (lambda r,e: _mean_or_nan(r['quality'][('antigen',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('antigen',e2)]])),
    'HLA(s) Allele':  (lambda r,e: _mean_or_nan(r['quality'][('allele',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('allele',e2)]])),
    'HLA(s) Eplets':  (lambda r,e: _mean_or_nan(r['quality'][('eplet',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('eplet',e2)]])),
    'Waiting Time':   (lambda r,e: _mean_or_nan(r['waiting_times_by_eth'][e]),
                       lambda r: _mean_or_nan([w for e2 in ETHCATS for w in r['waiting_times_by_eth'][e2]])),
    'HLA ClassI':     (lambda r,e: _mean_or_nan(r['quality'][('classI',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('classI',e2)]])),
    'HLA DR':         (lambda r,e: _mean_or_nan(r['quality'][('DR',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DR',e2)]])),
    'HLA DQ':         (lambda r,e: _mean_or_nan(r['quality'][('DQ',e)]),
                       lambda r: _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DQ',e2)]])),
}

def _paired_rank_tests(diffs):
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {
        'p_wilcoxon':  p_w, 'p_sign': p_s,
        'n_pos': n_pos, 'n_neg': n_neg,
        'n_used': int(len(diffs)),
        'median_diff': float(np.median(diffs)) if len(diffs) else float('nan'),
    }

def rank_test_metric(results_for_scenario, eth_fn, overall_fn, ethcats):
    overall_vals = np.array([overall_fn(r) for r in results_for_scenario], dtype=float)
    out = {}
    for e in ethcats:
        eth_vals = np.array([eth_fn(r, e) for r in results_for_scenario], dtype=float)
        mask = ~(np.isnan(eth_vals) | np.isnan(overall_vals))
        diffs = eth_vals[mask] - overall_vals[mask]
        out[e] = _paired_rank_tests(diffs)
    return out

rank_results_all = {}
for opt_res in RESOLUTIONS:
    rank_results_all[opt_res] = {}
    for metric_name, (eth_fn, overall_fn) in METRIC_EXTRACTORS.items():
        rank_results_all[opt_res][metric_name] = rank_test_metric(
            all_results[opt_res], eth_fn, overall_fn, TESTED_ETHCATS)

def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f

for opt_res in RESOLUTIONS:
    print(f"\n===== opt={opt_res} (Rawlsian / data_balance) — flags  [W = Wilcoxon p<.05, S = sign test p<.05] =====")
    header = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
    print(header); print('-' * len(header))
    for m in METRIC_EXTRACTORS:
        row = f"{m:24s}"
        for e in TESTED_ETHCATS:
            row += f"{('[' + _flags(rank_results_all[opt_res][m][e]) + ']'):>14s}"
        print(row)

In [ ]:
# SAVE EXCEL WITH SIGNIFICANCE FORMATTING (Rawlsian / data_balance)

from openpyxl import Workbook
from openpyxl.styles import Font

ALPHA = 0.05
METRIC_COLUMNS = list(METRIC_EXTRACTORS.keys())

wb = Workbook(); wb.remove(wb.active)
for opt_res in RESOLUTIONS:
    ws = wb.create_sheet(f'opt_{opt_res}')
    tbl = tables[opt_res]
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name); c.font = Font(bold=True)
    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_raw = row['Ethnicity(s)']
        is_eth = isinstance(eth_raw, (int, np.integer))
        eth_code = int(eth_raw) if is_eth else None
        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            if col_name == 'Ethnicity(s)' and is_eth:
                val = ETH_LABELS.get(eth_code, str(eth_raw))
            cell = ws.cell(row=row_pos, column=col_idx, value=val)
            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
                r = rank_results_all[opt_res][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)
    foot = len(tbl) + 4
    m = best_multipliers_per_scenario[opt_res]
    ws.cell(row=foot, column=1,
            value='Rawlsian equity weights used: ' + ', '.join(
                f'{ETH_LABELS[e]}={m[e]:.4f}' for e in EQUITY_ETHCATS))
    fb = ws.cell(row=foot+2, column=1, value='   bold      = Wilcoxon signed-rank p < 0.05 (primary)')
    fu = ws.cell(row=foot+3, column=1, value='   underlined = sign test p < 0.05 (robustness)')
    fb.font = Font(bold=True); fu.font = Font(underline='single')

out_path = RESULTS_DIR / f'results_databalance_full_10loci_3scenarios_rawlsian_significance{WARMUP_SUFFIX}.xlsx'
wb.save(out_path)
print(f'Saved: {out_path}')

In [ ]:
# Pairwise tests + marks en el XLSX
from openpyxl import Workbook
from openpyxl.styles import Font
import numpy as np


def _paired_rank_tests(diffs):
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {'p_wilcoxon': p_w, 'p_sign': p_s, 'n_pos': n_pos, 'n_neg': n_neg,
            'n_used': int(len(diffs)),
            'median_diff': float(np.median(diffs)) if len(diffs) else float('nan')}

def rank_test_metric_pair(results_A, results_B, eth_fn, ethcats):
    n = min(len(results_A), len(results_B))
    out = {}
    for e in ethcats:
        vals_A = np.array([eth_fn(results_A[i], e) for i in range(n)], dtype=float)
        vals_B = np.array([eth_fn(results_B[i], e) for i in range(n)], dtype=float)
        mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
        diffs = vals_A[mask] - vals_B[mask]
        out[e] = _paired_rank_tests(diffs)
    return out


rank_results_pairwise = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise[res_A][res_B] = {}
        for metric_name, (eth_fn, _) in METRIC_EXTRACTORS.items():
            rank_results_pairwise[res_A][res_B][metric_name] = rank_test_metric_pair(
                all_results[res_A], all_results[res_B], eth_fn, TESTED_ETHCATS)


def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f

print("\n===== PAIRWISE BETWEEN-RESOLUTION comparisons =====")
print("    [W = Wilcoxon p<.05, S = sign test p<.05]")
seen = set()
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        key = tuple(sorted([res_A, res_B]))
        if key in seen: continue
        seen.add(key)
        a, b = key
        print(f"\n--- {a} vs {b} ---")
        hdr = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
        print(hdr); print("-" * len(hdr))
        for m in METRIC_EXTRACTORS:
            row = f"{m:24s}"
            for e in TESTED_ETHCATS:
                row += f"{('[' + _flags(rank_results_pairwise[a][b][m][e]) + ']'):>14s}"
            print(row)


ALPHA = 0.05
METRIC_COLUMNS = list(METRIC_EXTRACTORS.keys())
OTHER_RESS = {opt_res: [r for r in RESOLUTIONS if r != opt_res] for opt_res in RESOLUTIONS}

rank_results_pairwise_overall = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise_overall[res_A][res_B] = {}
        for metric_name, (_, overall_fn) in METRIC_EXTRACTORS.items():
            A = all_results[res_A]; B = all_results[res_B]
            n = min(len(A), len(B))
            vals_A = np.array([overall_fn(A[i]) for i in range(n)], dtype=float)
            vals_B = np.array([overall_fn(B[i]) for i in range(n)], dtype=float)
            mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
            rank_results_pairwise_overall[res_A][res_B][metric_name] = _paired_rank_tests(vals_A[mask] - vals_B[mask])

def _pairwise_marks_overall(opt_res, metric):
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise_overall[opt_res][other][metric]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks
_SMALL = SMALL_ETHCATS if 'SMALL_ETHCATS' in dir() else set()

def _pairwise_marks(opt_res, metric, eth_code):
    if eth_code not in TESTED_ETHCATS: return ''
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise[opt_res][other][metric][eth_code]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

wb = Workbook(); wb.remove(wb.active)
for opt_res in RESOLUTIONS:
    ws = wb.create_sheet(f'opt_{opt_res}')
    tbl = tables[opt_res]
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name); c.font = Font(bold=True)
    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_raw = row['Ethnicity(s)']
        is_eth = isinstance(eth_raw, (int, np.integer))
        eth_code = int(eth_raw) if is_eth else None
        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            if col_name == 'Ethnicity(s)' and is_eth:
                val = ETH_LABELS.get(eth_code, str(eth_raw)) + ('*' if eth_code in _SMALL else '')
            cell = ws.cell(row=row_pos, column=col_idx, value=val)
            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
                r = rank_results_all[opt_res][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)
                marks = _pairwise_marks(opt_res, col_name, eth_code)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'
            if col_name in METRIC_COLUMNS and (eth_code is None):
                marks = _pairwise_marks_overall(opt_res, col_name)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'
    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)
    foot = len(tbl) + 4
    m_best = best_multipliers_per_scenario[opt_res]
    ws.cell(row=foot, column=1,
            value='Rawlsian equity weights used: ' + ', '.join(
                f'{ETH_LABELS[e]}={m_best[e]:.4f}' for e in EQUITY_ETHCATS))
    fb = ws.cell(row=foot+2, column=1, value='   bold      = Wilcoxon signed-rank p < 0.05 (eth vs overall, primary)')
    fu = ws.cell(row=foot+3, column=1, value='   underlined = sign test p < 0.05 (eth vs overall, robustness)')
    if len(OTHER_RESS[opt_res]) >= 1:
        o1 = OTHER_RESS[opt_res][0]
        ws.cell(row=foot+4, column=1, value=f'   †  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {o1} (paired by sim_id)')
    if len(OTHER_RESS[opt_res]) >= 2:
        o2 = OTHER_RESS[opt_res][1]
        ws.cell(row=foot+5, column=1, value=f'   ‡  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {o2} (paired by sim_id)')
    if _SMALL:
        ws.cell(row=foot+6, column=1, value='   (*) AmInd & PacIsl: small per-sim arrivals — interpret with caution.')
    fb.font = Font(bold=True); fu.font = Font(underline='single')


out_path = RESULTS_DIR / f'results_databalance_full_10loci_3scenarios_rawlsian_significance{WARMUP_SUFFIX}.xlsx'


wb.save(out_path)
print(f'\n✓ Saved: {out_path}')